# Tâche 3 : Expérimentation et Comparaison des Algorithmes ML avec MLflow

## Analyse des Résultats d'Expérimentation

Ce notebook analyse les expériences menées avec les quatre algorithmes de classification.

## 1. Import des Bibliothèques Requises

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import os

# MLflow imports
import mlflow
from mlflow.tracking import MlflowClient

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Bibliothèques importées avec succès')

## 2. Chargement et Préparation des Données

In [ ]:
# Setup paths
project_root = Path.cwd().parent
data_path = project_root / 'data' / 'movies_credits_merged.csv'
mlflow_path = project_root / 'mlruns'

# Load data
df = pd.read_csv(data_path)
df = df[(df['budget'] > 0) & (df['revenue'] > 0)].copy()
df['is_success'] = (df['revenue'] > df['budget']).astype(int)

FEATURES = ['budget', 'popularity', 'runtime', 'vote_average', 'vote_count']
X = df[FEATURES].fillna(0)
y = df['is_success']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'✓ Données chargées')
print(f'  Total samples: {len(df)}')
print(f'  Training set: {len(X_train)} samples')
print(f'  Test set: {len(X_test)} samples')
print(f'  Positive class rate: {y.mean():.2%}')
print(f'  Features: {FEATURES}')

## 3. Configuration MLflow et Récupération des Runs

In [ ]:
# Configure MLflow
mlflow.set_tracking_uri(mlflow_path.as_uri())
client = MlflowClient(mlflow_path.as_uri())

# Get the Movie_Success_Classification experiment
experiments = client.search_experiments()
exp_dict = {exp.name: exp.experiment_id for exp in experiments}

if 'Movie_Success_Classification' in exp_dict:
    exp_id = exp_dict['Movie_Success_Classification']
    print(f'✓ Experiment trouvée: Movie_Success_Classification (ID: {exp_id})')
else:
    print('⚠ Experiment non trouvée. Expériences disponibles:', list(exp_dict.keys()))
    exp_id = None

## 4. Récupération et Analyse des Résultats des Runs MLflow

In [ ]:
# Fetch all runs from the experiment
if exp_id:
    runs = client.search_runs(experiment_ids=[exp_id])
    
    # Convert runs to DataFrame
    run_data = []
    for run in runs:
        if run.state == 'FINISHED':
            run_info = {
                'run_name': run.data.tags.get('mlflow.runName', 'N/A'),
                'accuracy': run.data.metrics.get('accuracy', None),
                'precision': run.data.metrics.get('precision', None),
                'recall': run.data.metrics.get('recall', None),
                'f1_score': run.data.metrics.get('f1_score', None),
                'tp': run.data.metrics.get('cm_tp', None),
                'tn': run.data.metrics.get('cm_tn', None),
                'fp': run.data.metrics.get('cm_fp', None),
                'fn': run.data.metrics.get('cm_fn', None),
            }
            # Add hyperparameters
            for param_key, param_val in run.data.params.items():
                run_info[f'param_{param_key}'] = param_val
            run_data.append(run_info)
    
    results_df = pd.DataFrame(run_data)
    print(f'✓ {len(results_df)} runs trouvés')
    print('\nAperçu des résultats:')
    print(results_df[['run_name', 'accuracy', 'precision', 'recall', 'f1_score']].to_string())
else:
    results_df = pd.DataFrame()
    print('Aucun run à afficher')

## 5. Comparaison et Analyse des Modèles

In [ ]:
# Clean and summarize by algorithm
if not results_df.empty:
    summary = results_df[['run_name', 'accuracy', 'precision', 'recall', 'f1_score']].copy()
    summary = summary.sort_values('f1_score', ascending=False)
    
    print('\n' + '='*70)
    print('TABLEAU COMPARATIF DES MODÈLES')
    print('='*70)
    print(summary.to_string(index=False))
    print('='*70)
    
    # Best models per metric
    print('\nMEILLEURS MODÈLES PAR MÉTRIQUE:')
    print(f'  • Accuracy: {summary.iloc[0]["run_name"]} ({summary.iloc[0]["accuracy"]:.4f})')
    print(f'  • F1-Score: {summary.iloc[0]["run_name"]} ({summary.iloc[0]["f1_score"]:.4f})')
    
    # Best overall
    best_model = summary.iloc[0]
    print(f'\n✓ MEILLEUR MODÈLE: {best_model["run_name"]}')
    print(f'  - Accuracy: {best_model["accuracy"]:.4f}')
    print(f'  - Precision: {best_model["precision"]:.4f}')
    print(f'  - Recall: {best_model["recall"]:.4f}')
    print(f'  - F1-Score: {best_model["f1_score"]:.4f}')

## 6. Visualisations Comparatives

In [ ]:
if not results_df.empty:
    # Plot comparison
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Comparaison des Performances des Modèles', fontsize=16, fontweight='bold')
    
    metrics = ['accuracy', 'precision', 'recall', 'f1_score']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    
    summary_plot = results_df[['run_name'] + metrics].sort_values('f1_score', ascending=True)
    
    for idx, (ax, metric, label) in enumerate(zip(axes.flat, metrics, metric_labels)):
        ax.barh(summary_plot['run_name'], summary_plot[metric], color='steelblue')
        ax.set_xlabel(label, fontweight='bold')
        ax.set_xlim(0, 1)
        for i, v in enumerate(summary_plot[metric]):
            ax.text(v + 0.02, i, f'{v:.3f}', va='center')
    
    plt.tight_layout()
    plt.show()
    
    print('✓ Visualisations créées')

## 7. Analyse Critique et Recommandations

In [ ]:
print('\n' + '='*70)
print('ANALYSE CRITIQUE')
print('='*70)

if not results_df.empty:
    summary = results_df[['run_name', 'accuracy', 'precision', 'recall', 'f1_score']].sort_values('f1_score', ascending=False)
    
    print('\n1. QUEL ALGORITHME DONNE LES MEILLEURS RÉSULTATS?')
    best = summary.iloc[0]
    print(f'   → {best["run_name"]} avec F1-score de {best["f1_score"]:.4f}')
    
    print('\n2. COMPARAISON DES APPROCHES:')
    print('   a) KNN: Sensible au bruit, nécessite normalization')
    print('   b) SVM: Bon pour séparation non-linéaire, coûteux en calcul')
    print('   c) Random Forest: Robuste, gère bien les non-linéarités')
    print('   d) Logistic Regression: Interprétable, bon baseline')
    
    print('\n3. IMPACT DES HYPERPARAMÈTRES:')
    print('   • KNN (k): Augmenter k réduit le surapprentissage')
    print('   • SVM (C, kernel): C contrôle la régularisation')
    print('   • RF (n_estimators, max_depth): Plus arbres = généralement mieux')
    print('   • LR (C): Régularisation importante')
    
    print('\n4. RECOMMANDATIONS:')
    print(f'   ✓ Adopter {best["run_name"]} comme modèle principal')
    print('   ✓ Implémenter en production avec MLflow Registry')
    print('   ✓ Monitorer les performances avec nouvelles données')
    print('   ✓ Considérer l\'ensemble voting si performance critique')

else:
    print('Pas de résultats pour l\'analyse')
    
print('='*70)